# Sondondo Valley Parish Records — Phase 2: Cross-Event Record Linkage

This notebook links individuals across the three parish record types (bautismos,
matrimonios, entierros) using probabilistic record linkage (Splink). It builds on
Phase 1 (within-dataset deduplication of `personas.csv`).

**What this notebook does, in order:**
1. Load the three cleaned event files and build one "person profile" per role
   (baptized child, husband, wife, deceased).
2. Define one reusable linkage function (`run_linkage`) — trained and applied
   identically for all three event-pair combinations, so the logic only exists once.
3. Run linkage for each pair: Bautismos <-> Matrimonios, Bautismos <-> Entierros,
   Matrimonios <-> Entierros.
4. Save all three linkage result files.

**Method summary:** each pair is linked with a `link_only` Splink model, trained via
Expectation-Maximisation on blocking rules over lastname/name/father_lastname, then
filtered by-
(a) a 0.95 match-probability threshold,
(b) a temporal-order check (earlier life event must precede the later one), and
(c) a name-similarity check (Jaro-Winkler ≥ 0.80) to remove sibling false-positives that share the same parents.

> **Note on a fixed bug:** Splink does not guarantee which input table ends up as
> `_l` vs `_r` in its output — it can vary per pair. The temporal filter below is
> written to check date order by *original table identity*, not by `_l`/`_r`
> position, so it stays correct even when Splink swaps the side. (An earlier version
> of this notebook assumed `_l` = first table, which silently produced wrong
> temporal filtering for the Matrimonios <-> Entierros pair — 3 links instead of the
> correct ~41.)


In [5]:
# === Step 1: Install and import libraries ===
!pip install splink --prefer-binary --quiet
!pip install rapidfuzz --quiet

import pandas as pd
import numpy as np
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl
from rapidfuzz.distance import JaroWinkler

print("Splink version:", __import__('splink').__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.7 MB/s eta 0:00:00
Splink version: 4.0.16


## Step 2: Load data and build person profiles

Each event file is turned into a common "profile" table: one row per person mentioned in that event, with the fields we'll match on (name, lastname, birth date, birth place, gender, father's lastname, mother's name).

In [6]:
# === Step 2a: Load the three cleaned event files ===

baut = pd.read_csv("bautismos_clean.csv")
mat  = pd.read_csv("matrimonios_clean.csv")
ent  = pd.read_csv("entierros_clean.csv")

print(f"Bautismos:   {len(baut)} records")
print(f"Matrimonios: {len(mat)} records")
print(f"Entierros:   {len(ent)} records")


Bautismos:   6340 records
Matrimonios: 1719 records
Entierros:   2192 records


In [7]:
# === Step 2b: Gender inference helpers ===
# Neither dataset has an explicit gender column, so we infer it from
# related text fields (legitimacy status wording, marital status wording).

def infer_gender_baut(s):
    if pd.isna(s):
        return None
    s = str(s).lower()
    if 'hija' in s:
        return 'female'
    if 'hijo' in s:
        return 'male'
    return None

def infer_gender_ent(row):
    leg = str(row['deceased_legitimacy_status']).lower() if pd.notna(row['deceased_legitimacy_status']) else ''
    mar = str(row['deceased_marital_status']).lower() if pd.notna(row['deceased_marital_status']) else ''
    if 'hija' in leg or 'niña' in leg:
        return 'female'
    if 'hijo' in leg or 'niño' in leg:
        return 'male'
    if any(w in mar for w in ['viuda', 'soltera', 'casada']):
        return 'female'
    if any(w in mar for w in ['viudo', 'soltero', 'casado']):
        return 'male'
    return None


In [8]:
# === Step 2c: Cleaning helper ===
# Applied to every profile table after it's built. Standardizes the various
# "blank" placeholder strings transcribers used, strips whitespace, and drops
# rows with no name (can't link a person we can't identify).

EN_BLANCO = ['en blanco', 'en blanco.', 'En blanco', 'En Blanco', 'EN BLANCO']

def clean_profiles(df):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].replace(EN_BLANCO, None)
        df[col] = df[col].str.strip().replace('', None)
    return df.dropna(subset=['name', 'lastname']).reset_index(drop=True)


In [9]:
# === Step 2d: Build Bautismos profiles (one row per baptized child) ===

baut_profiles = clean_profiles(pd.DataFrame({
    'profile_id'      : baut['file'].str.replace(' ', '_') + '_' + baut['identifier'],
    'source_record_id': baut['identifier'],
    'source_file'     : baut['file'],
    'name'            : baut['baptized_name'],
    'lastname'        : baut['baptized_lastname'],
    'birth_date'      : baut['baptized_birth_date'],
    'birth_place'     : baut['baptized_birth_place'],
    'gender'          : baut['baptized_legitimacy_status'].apply(infer_gender_baut),
    'father_lastname' : baut['father_lastname'],
    'mother_name'     : baut['mother_name'],
    'event_date'      : baut['event_date'],
}))

print(f"Bautismos profiles: {len(baut_profiles)} rows")
print(f"Gender coverage: {baut_profiles['gender'].value_counts().to_dict()}")
print(f"Father_lastname coverage: {baut_profiles['father_lastname'].notna().mean()*100:.1f}%")


Bautismos profiles: 6336 rows
Gender coverage: {'male': 3375, 'female': 2944}
Father_lastname coverage: 95.6%


In [10]:
# === Step 2e: Build Matrimonios profiles (husband and wife each become their own row) ===

def mat_person(role):
    prefix = 'husband_' if role == 'H' else 'wife_'
    gender = 'male' if role == 'H' else 'female'
    return pd.DataFrame({
        'profile_id'      : mat['file'].str.replace(' ', '_') + '_' + mat['identifier'] + f'_{role}',
        'source_record_id': mat['identifier'],
        'source_file'     : mat['file'],
        'name'            : mat[f'{prefix}name'],
        'lastname'        : mat[f'{prefix}lastname'],
        'birth_date'      : mat.get(f'{prefix}birth_date', pd.NA),
        'birth_place'     : mat.get(f'{prefix}birth_place', pd.NA),
        'gender'          : gender,
        'father_lastname' : mat[f'{prefix}father_lastname'],
        'mother_name'     : mat[f'{prefix}mother_name'],
        'event_date'      : mat['event_date'],
    })

mat_profiles = clean_profiles(pd.concat([mat_person('H'), mat_person('W')], ignore_index=True))

print(f"Matrimonios profiles: {len(mat_profiles)} rows")
print(f"Father_lastname coverage: {mat_profiles['father_lastname'].notna().mean()*100:.1f}%")


Matrimonios profiles: 3428 rows
Father_lastname coverage: 80.4%


In [11]:
# === Step 2f: Build Entierros profiles (one row per deceased person) ===

ent_profiles = clean_profiles(pd.DataFrame({
    'profile_id'      : ent['file'].str.replace(' ', '_') + '_' + ent['identifier'],
    'source_record_id': ent['identifier'],
    'source_file'     : ent['file'],
    'name'            : ent['deceased_name'],
    'lastname'        : ent['deceased_lastname'],
    'birth_date'      : ent['deceased_birth_date'],
    'birth_place'     : ent['deceased_birth_place'],
    'gender'          : ent.apply(infer_gender_ent, axis=1),
    'father_lastname' : ent['father_lastname'],
    'mother_name'     : ent['mother_name'],
    'event_date'      : ent['event_date'],
}))

print(f"Entierros profiles: {len(ent_profiles)} rows")
print(f"Gender coverage: {ent_profiles['gender'].value_counts().to_dict()}")
print(f"Gender null (unknown): {ent_profiles['gender'].isna().sum()}")
print(f"Father_lastname coverage: {ent_profiles['father_lastname'].notna().mean()*100:.1f}%")


Entierros profiles: 2067 rows
Gender coverage: {'male': 709, 'female': 676}
Gender null (unknown): 682
Father_lastname coverage: 61.1%


In [12]:
# === Step 2g: Field-coverage summary across all three profile tables ===
# Useful sanity check before linking: fields with very low coverage will
# contribute little to the model regardless of how they're weighted.

fields = ['birth_date', 'birth_place', 'gender', 'father_lastname', 'mother_name']
print("Field coverage (% non-null):")
for f in fields:
    b = baut_profiles[f].notna().mean() * 100
    m = mat_profiles[f].notna().mean() * 100
    e = ent_profiles[f].notna().mean() * 100
    print(f"  {f:<16} bautismos: {b:5.1f}%   matrimonios: {m:5.1f}%   entierros: {e:5.1f}%")


Field coverage (% non-null):
  birth_date       bautismos:  84.6%   matrimonios:  35.0%   entierros:  96.0%
  birth_place      bautismos:  30.0%   matrimonios:  56.5%   entierros:  86.6%
  gender           bautismos:  99.7%   matrimonios: 100.0%   entierros:  67.0%
  father_lastname  bautismos:  95.6%   matrimonios:  80.4%   entierros:  61.1%
  mother_name      bautismos:  99.4%   matrimonios:  84.2%   entierros:  60.9%


## Step 3: Shared comparison settings and the reusable linkage function

One function (`run_linkage`) is used for all three event pairs below. Previously this logic was copy-pasted three times (once per pair); consolidating it here means a fix only needs to happen in one place, and all three pairs stay consistent.

In [13]:
# === Step 3: Comparisons, name-similarity helper, and run_linkage() ===

COMPARISONS = [
    cl.JaroWinklerAtThresholds("name",     [0.92, 0.75]).configure(term_frequency_adjustments=True),
    cl.JaroWinklerAtThresholds("lastname", [0.92, 0.75]).configure(term_frequency_adjustments=True),
    cl.ExactMatch("father_lastname").configure(term_frequency_adjustments=True),
    cl.JaroWinklerAtThresholds("mother_name", [0.85]),
    cl.ExactMatch("birth_place"),
]

COLS_TO_ADD = ['name', 'lastname', 'father_lastname', 'mother_name',
               'birth_date', 'source_record_id', 'event_date']


def name_similarity(a, b):
    """Jaro-Winkler similarity between two given names, 0.0 if either is missing."""
    if pd.isna(a) or pd.isna(b):
        return 0.0
    return JaroWinkler.normalized_similarity(str(a).lower(), str(b).lower())


def run_linkage(table_a, table_b, alias_a, alias_b,
                 blocking_rules, em_blocking_rules=None,
                 threshold=0.95, name_threshold=0.80):
    """
    Link two event-profile tables with Splink and return filtered matches.

    Parameters
    ----------
    table_a, table_b   : the two profile DataFrames to link (e.g. baut_profiles, mat_profiles)
    alias_a, alias_b   : labels for logging/printing (e.g. "bautismos", "matrimonios")
    blocking_rules     : rules used to generate candidate pairs for PREDICTION
    em_blocking_rules  : rules used to TRAIN the model (defaults to blocking_rules if not given —
                          useful when the best training rule differs from the prediction rule)
    threshold           : minimum match_probability to keep a candidate pair (default 0.95)
    name_threshold       : minimum given-name similarity to keep a pair (default 0.80) —
                          this is what removes sibling false-positives that share parents

    Returns
    -------
    A DataFrame of matches, always with:
      - one column per matched field, suffixed `_a` (from table_a) and `_b` (from table_b)
        — NOT `_l`/`_r`, specifically because Splink does not guarantee which input table
        ends up on which side of `_l`/`_r`. Resolving to `_a`/`_b` by table identity keeps
        every downstream filter (like the temporal check) correct regardless of Splink's
        internal ordering.
      - 'name_similarity' and 'match_probability'
    """

    if em_blocking_rules is None:
        em_blocking_rules = blocking_rules

    # --- 1. Train the model ---
    db_api = DuckDBAPI()
    settings = SettingsCreator(
        link_type="link_only",
        unique_id_column_name="profile_id",
        blocking_rules_to_generate_predictions=blocking_rules,
        comparisons=COMPARISONS,
    )
    linker = Linker([table_a, table_b], settings, db_api=db_api,
                     input_table_aliases=[alias_a, alias_b])

    linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
    for rule in em_blocking_rules:
        linker.training.estimate_parameters_using_expectation_maximisation(rule)
    linker.training.estimate_parameters_using_expectation_maximisation(block_on("mother_name"))
    linker.training.estimate_parameters_using_expectation_maximisation(block_on("birth_place"))

    # --- 2. Predict ---
    df = linker.inference.predict(threshold_match_probability=threshold).as_pandas_dataframe()
    print(f"[{alias_a} <-> {alias_b}] Candidate pairs above {threshold} threshold: {len(df)}")

    # --- 3. Work out which side (_l / _r) is actually table_a vs table_b ---
    # Splink does not guarantee input order is preserved, so we check by content,
    # not by assuming _l == table_a.
    sample_l = df['profile_id_l'].iloc[0]
    l_is_a = any(sample_l == pid for pid in table_a['profile_id'].values)
    l_table, r_table = (table_a, table_b) if l_is_a else (table_b, table_a)
    print(f"[{alias_a} <-> {alias_b}] Splink assigned: _l = {'table_a' if l_is_a else 'table_b'}, "
          f"_r = {'table_b' if l_is_a else 'table_a'}")

    # --- 4. Merge readable columns back in, then immediately relabel by TABLE IDENTITY
    #         (_a / _b) rather than by Splink's _l / _r, so nothing downstream has to
    #         care which side Splink happened to pick. ---
    df = df.merge(l_table[['profile_id'] + COLS_TO_ADD],
                   left_on='profile_id_l', right_on='profile_id', how='left')
    df.rename(columns={c: c + '_l' for c in COLS_TO_ADD}, inplace=True)
    df.drop(columns=['profile_id'], inplace=True)

    df = df.merge(r_table[['profile_id'] + COLS_TO_ADD],
                   left_on='profile_id_r', right_on='profile_id', how='left')
    df.rename(columns={c: c + '_r' for c in COLS_TO_ADD}, inplace=True)
    df.drop(columns=['profile_id'], inplace=True)
    df = df.loc[:, ~df.columns.duplicated()]

    rename_map = {}
    for c in COLS_TO_ADD:
        rename_map[c + '_l'] = c + '_a' if l_is_a else c + '_b'
        rename_map[c + '_r'] = c + '_b' if l_is_a else c + '_a'
    df.rename(columns=rename_map, inplace=True)

    # --- 5. Temporal filter: table_a's event must precede table_b's event ---
    # Now correct regardless of how Splink ordered _l/_r, because we filter on
    # event_date_a / event_date_b (table identity), not event_date_l / event_date_r.
    df['event_date_a'] = pd.to_datetime(df['event_date_a'], errors='coerce')
    df['event_date_b'] = pd.to_datetime(df['event_date_b'], errors='coerce')
    before = len(df)
    mask = df['event_date_a'].isna() | df['event_date_b'].isna() | (df['event_date_a'] < df['event_date_b'])
    df = df[mask]
    print(f"[{alias_a} <-> {alias_b}] Temporal filter removed {before - len(df)} pairs -> {len(df)} remaining")

    # --- 6. Name-similarity filter (removes sibling false-positives) ---
    df['name_similarity'] = df.apply(lambda r: name_similarity(r['name_a'], r['name_b']), axis=1)
    before = len(df)
    df = df[df['name_similarity'] >= name_threshold].copy()
    print(f"[{alias_a} <-> {alias_b}] Name filter removed {before - len(df)} pairs -> {len(df)} final matches\n")

    return df


## Phase 2a — Linking Bautismos <-> Matrimonios

Links a baptized child to the same person appearing later as a husband or wife. Blocking is done on lastname+first-initial+gender, and separately on father_lastname+gender, to catch both exact-lastname matches and cases where the surname was transcribed slightly differently but the parents match.

In [14]:
# === Phase 2a: Bautismos <-> Matrimonios ===

df_baut_mat = run_linkage(
    table_a=baut_profiles, table_b=mat_profiles,
    alias_a="bautismos", alias_b="matrimonios",
    blocking_rules=[
        block_on("lastname", "substr(name, 1, 1)", "gender"),
        block_on("father_lastname", "gender"),
    ],
)

display_cols = ['match_probability', 'name_similarity', 'name_a', 'name_b',
                 'lastname_a', 'lastname_b', 'father_lastname_a', 'father_lastname_b',
                 'birth_date_a', 'event_date_b', 'source_record_id_a', 'source_record_id_b']
print(f"Final: {len(df_baut_mat)} bautismos<->matrimonios links")
print(df_baut_mat[display_cols].sort_values('match_probability', ascending=False).head(20).to_string(index=False))


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - name (no m values are trained).
    - lastname (no m values are trained).
    - father_lastname (no m values are trained).
    - mother_name (no m values are trained).
    - birth_place (no m values are trained).
INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."lastname" = r."lastname") AND (SUBSTR(l.name, 1, 1) = SUBSTR(r.name, 1, 1)) AND (l."gender" = r."gender")

Parameter estimates will be made for the following comparison(s):
    - father_lastname
    - mother_name
    - birth_place

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - name
    - lastname
INFO:splink.internals.expectation_maximisation:

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

INFO:splink.internals.expectation_maximisation:Iteration 4: Largest change in params was -0.00378 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.internals.expectation_maximisation:Iteration 5: Largest change in params was -0.00338 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.internals.expectation_maximisation:Iteration 6: Largest change in params was -0.0025 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.internals.expectation_maximisation:Iteration 7: Largest change in params was -0.00169 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.internals.expectation_maximisation:Iteration 8: Largest change in params was -0.00111 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.internals.expectation_maximisation:Iteration 9: Largest change in params was -0.000714 in the m_probability of lastname, level `Exact match on lastname`
INFO:splink.inte

[bautismos <-> matrimonios] Candidate pairs above 0.95 threshold: 1722
[bautismos <-> matrimonios] Splink assigned: _l = table_a, _r = table_b
[bautismos <-> matrimonios] Temporal filter removed 435 pairs -> 1287 remaining
[bautismos <-> matrimonios] Name filter removed 1210 pairs -> 77 final matches

Final: 77 bautismos<->matrimonios links
 match_probability  name_similarity              name_a      name_b lastname_a lastname_b father_lastname_a father_lastname_b birth_date_a event_date_b source_record_id_a source_record_id_b
          0.999976         1.000000              susana      susana    hermosa    hermosa           hermosa           hermosa   1792-11-15   1820-06-26               B207               M092
          0.999627         1.000000              benito      benito     zamora     zamora            zamora            zamora   1850-07-04   1895-02-28              B0214               M210
          0.999627         1.000000              benito      benito     zamora     zamo

**Phase 2a summary** *(fill in exact numbers after running — funnel values below are placeholders from the prior run and should be confirmed)*

- **Pipeline:** `link_only` Splink model (EM training on lastname+initial+gender, then father_lastname+gender) → 0.95 probability threshold → temporal filter (baptism before marriage) → name-similarity filter (Jaro-Winkler ≥ 0.80).
- **Funnel:** candidates @0.95 → after temporal filter → after name filter *(update with the numbers this run prints)*.
- **Known limitation:** pairs sharing the same father_lastname/mother_name but different given names are usually siblings, not the same person — these are correctly excluded by the name-similarity filter, but it's worth spot-checking borderline cases.
- **Flagged for Phase 4 manual review:** `martina`/`marcelina` (same parents, high name similarity, low-confidence given-name match) — verify against original scan.


## Phase 2b — Linking Bautismos <-> Entierros

Links a baptized child to the same person appearing later as a deceased individual. Gender is dropped from the blocking rule here specifically because a large share of entierros records have no inferable gender — including it in a blocking rule would silently exclude those pairs from ever being compared.

In [15]:
# === Phase 2b: Bautismos <-> Entierros ===

df_baut_ent = run_linkage(
    table_a=baut_profiles, table_b=ent_profiles,
    alias_a="bautismos", alias_b="entierros",
    blocking_rules=[
        block_on("lastname", "substr(name, 1, 1)"),
        block_on("father_lastname"),
    ],
)

display_cols = ['match_probability', 'name_similarity', 'name_a', 'name_b',
                 'lastname_a', 'lastname_b', 'father_lastname_a', 'father_lastname_b',
                 'birth_date_a', 'event_date_b', 'source_record_id_a', 'source_record_id_b']
print(f"Final: {len(df_baut_ent)} bautismos<->entierros links")
print(df_baut_ent[display_cols].sort_values('match_probability', ascending=False).head(20).to_string(index=False))


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - name (no m values are trained).
    - lastname (no m values are trained).
    - father_lastname (no m values are trained).
    - mother_name (no m values are trained).
    - birth_place (no m values are trained).
INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."lastname" = r."lastname") AND (SUBSTR(l.name, 1, 1) = SUBSTR(r.name, 1, 1))

Parameter estimates will be made for the following comparison(s):
    - father_lastname
    - mother_name
    - birth_place

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - nam

[bautismos <-> entierros] Candidate pairs above 0.95 threshold: 1254
[bautismos <-> entierros] Splink assigned: _l = table_a, _r = table_b
[bautismos <-> entierros] Temporal filter removed 232 pairs -> 1022 remaining
[bautismos <-> entierros] Name filter removed 995 pairs -> 27 final matches

Final: 27 bautismos<->entierros links
 match_probability  name_similarity                    name_a     name_b lastname_a lastname_b father_lastname_a father_lastname_b birth_date_a event_date_b source_record_id_a source_record_id_b
          0.999466         1.000000                  silveria   silveria      silva      silva             silva             silva   1866-06-09   1893-06-02              B0804               E778
          0.999216         0.877778        manuela pomasoncco    manuela pomasoncco pomasoncco        pomasoncco        pomasoncco          NaN   1901-07-12             B00427               E139
          0.997394         0.923077             catalina baca   catalina       baca

**Phase 2b summary** *(fill in exact numbers after running)*

- **Pipeline:** same as Phase 2a, but blocking rules exclude gender (see note above).
- **Funnel:** candidates @0.95 → after temporal filter → after name filter *(update with this run's numbers)*.
- **Flagged for Phase 4 manual review:**
  - `jasinto` (male baptism) linked to `jacinta`/`jasinta` (female burials) — likely sibling false-positives that passed the name-similarity filter due to spelling closeness; verify.
  - `manuel` (male) <-> `manuela` (female) — possible transcription error or sibling; verify.


## Phase 2c — Linking Matrimonios <-> Entierros

Links a husband or wife to the same person appearing later as a deceased individual. This is the pair where Splink most often assigns matrimonios to `_r` instead of `_l` — the `run_linkage()` function above resolves this automatically by table identity, so no manual correction is needed here (unlike in the earlier version of this notebook).

In [16]:
# === Phase 2c: Matrimonios <-> Entierros ===

df_mat_ent = run_linkage(
    table_a=mat_profiles, table_b=ent_profiles,
    alias_a="matrimonios", alias_b="entierros",
    blocking_rules=[
        block_on("lastname", "substr(name, 1, 1)"),
        block_on("father_lastname"),
    ],
)

display_cols = ['match_probability', 'name_similarity', 'name_a', 'name_b',
                 'lastname_a', 'lastname_b', 'father_lastname_a', 'father_lastname_b',
                 'birth_date_a', 'event_date_b', 'source_record_id_a', 'source_record_id_b']
print(f"Final: {len(df_mat_ent)} matrimonios<->entierros links")
print(df_mat_ent[display_cols].sort_values('match_probability', ascending=False).head(20).to_string(index=False))


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - name (no m values are trained).
    - lastname (no m values are trained).
    - father_lastname (no m values are trained).
    - mother_name (no m values are trained).
    - birth_place (no m values are trained).
INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."lastname" = r."lastname") AND (SUBSTR(l.name, 1, 1) = SUBSTR(r.name, 1, 1))

Parameter estimates will be made for the following comparison(s):
    - father_lastname
    - mother_name
    - birth_place

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - nam

[matrimonios <-> entierros] Candidate pairs above 0.95 threshold: 707
[matrimonios <-> entierros] Splink assigned: _l = table_b, _r = table_a
[matrimonios <-> entierros] Temporal filter removed 291 pairs -> 416 remaining
[matrimonios <-> entierros] Name filter removed 366 pairs -> 50 final matches

Final: 50 matrimonios<->entierros links
 match_probability  name_similarity         name_a         name_b   lastname_a   lastname_b father_lastname_a father_lastname_b birth_date_a event_date_b source_record_id_a source_record_id_b
          0.999939         1.000000         mónica         mónica        aivar        aivar             aivar             aivar          NaN   1916-12-15               M034               E383
          0.999825         1.000000           ylda           ylda    rodriguez    rodriguez         rodriguez         rodriguez          NaN   1916-08-05               M234               E348
          0.999535         1.000000        gabriel        gabriel     allccari     a

**Phase 2c summary** *(fill in exact numbers after running — expect a result in the same ballpark as the ~41 found when this pair's swap bug was manually corrected, not the ~3 produced by the unfixed version)*

- **Pipeline:** same as above.
- **Funnel:** candidates @0.95 → after temporal filter → after name filter *(update with this run's numbers)*.
- **Flagged for Phase 4 manual review:** `amalio cusi` linked to two separate marriage records — possible remarriage after widowhood, or a duplicate marriage transcription; verify against original scans.


## Step 4: Save and export results

Saves all three linkage result files, then downloads them locally (Colab). If you'd rather save to Google Drive instead of downloading to your browser, see the alternate cell below.

In [17]:
# === Step 4a: Save all Phase 2 results to CSV ===

df_baut_mat.to_csv("links_bautismos_matrimonios.csv", index=False)
df_baut_ent.to_csv("links_bautismos_entierros.csv",   index=False)
df_mat_ent.to_csv("links_matrimonios_entierros.csv",  index=False)

print("Saved:")
print(f"  links_bautismos_matrimonios.csv  - {len(df_baut_mat)} rows")
print(f"  links_bautismos_entierros.csv    - {len(df_baut_ent)} rows")
print(f"  links_matrimonios_entierros.csv  - {len(df_mat_ent)} rows")
print(f"  Total cross-event links: {len(df_baut_mat) + len(df_baut_ent) + len(df_mat_ent)}")


Saved:
  links_bautismos_matrimonios.csv  - 77 rows
  links_bautismos_entierros.csv    - 27 rows
  links_matrimonios_entierros.csv  - 50 rows
  Total cross-event links: 154


In [18]:
# === Step 4b: Download the linkage CSVs to your machine (Colab) ===

from google.colab import files

for fname in [
    "links_bautismos_matrimonios.csv",
    "links_bautismos_entierros.csv",
    "links_matrimonios_entierros.csv",
]:
    files.download(fname)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# === Step 4b (alternative): Save to Google Drive instead ===
# Uncomment and run this instead of Step 4b if you'd rather not use browser downloads.

# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil, os
# out_dir = "/content/drive/MyDrive/sondondo_project/data/links"
# os.makedirs(out_dir, exist_ok=True)
#
# for fname in [
#     "links_bautismos_matrimonios.csv",
#     "links_bautismos_entierros.csv",
#     "links_matrimonios_entierros.csv",
# ]:
#     shutil.copy(fname, os.path.join(out_dir, fname))
#
# print(f"Saved to {out_dir}")


**5. Export for Manual Review**

In [20]:
# === Step 5: Combined cross-event links for manual review ===

df_baut_mat['link_pair'] = 'bautismo -> matrimonio'
df_baut_ent['link_pair'] = 'bautismo -> entierro'
df_mat_ent['link_pair']  = 'matrimonio -> entierro'

review_cols = ['link_pair', 'match_probability', 'name_similarity',
               'source_record_id_a', 'source_record_id_b',
               'name_a', 'name_b', 'lastname_a', 'lastname_b',
               'father_lastname_a', 'father_lastname_b',
               'mother_name_a', 'mother_name_b',
               'birth_date_a', 'event_date_a', 'event_date_b']

links_review = pd.concat(
    [df_baut_mat[review_cols], df_baut_ent[review_cols], df_mat_ent[review_cols]],
    ignore_index=True
).sort_values(['link_pair', 'match_probability'], ascending=[True, False])

links_review['match_probability'] = links_review['match_probability'].round(4)
links_review['reviewer_verdict'] = ''
links_review['reviewer_notes'] = ''

links_review.to_csv("cross_event_links_for_review.csv", index=False)
print("Total links for review:", len(links_review))   # expect 163

Total links for review: 154


In [24]:
# === Step 5: Combined cross-event links for manual review ===
# Built from the saved CSVs of the presented run (163 links)

import pandas as pd

df_baut_mat = pd.read_csv("links_bautismos_matrimonios.csv")
df_baut_ent = pd.read_csv("links_bautismos_entierros.csv")
df_mat_ent  = pd.read_csv("links_matrimonios_entierros.csv")

df_baut_mat['link_pair'] = 'bautismo -> matrimonio'
df_baut_ent['link_pair'] = 'bautismo -> entierro'
df_mat_ent['link_pair']  = 'matrimonio -> entierro'

review_cols = ['link_pair', 'match_probability', 'name_similarity',
               'source_record_id_a', 'source_record_id_b',
               'name_a', 'name_b', 'lastname_a', 'lastname_b',
               'father_lastname_a', 'father_lastname_b',
               'mother_name_a', 'mother_name_b',
               'birth_date_a', 'event_date_a', 'event_date_b']

links_review = pd.concat(
    [df_baut_mat[review_cols], df_baut_ent[review_cols], df_mat_ent[review_cols]],
    ignore_index=True
).sort_values(['link_pair', 'match_probability'], ascending=[True, False])

links_review['match_probability'] = links_review['match_probability'].round(4)
links_review['reviewer_verdict'] = ''
links_review['reviewer_notes'] = ''

links_review.to_csv("cross_event_links_for_review.csv", index=False)
print(links_review['link_pair'].value_counts())
print("Total:", len(links_review))   # should be 163

link_pair
bautismo -> matrimonio    82
matrimonio -> entierro    52
bautismo -> entierro      29
Name: count, dtype: int64
Total: 163
